In [1]:
# ===== Cell 1: Install & Imports =====
#!pip install mediapipe

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, LSTM, TimeDistributed, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import Callback, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import mediapipe as mp
import seaborn as sns
from matplotlib import cm


In [2]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ===== Cell 2: GPU Setup & Configs =====
# GPU Setup
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"✅ GPU terdeteksi: {physical_devices[0].name}")
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
else:
    print("❌ GPU tidak tersedia. Menggunakan CPU.")

# Konstanta & Paths
IMAGE_HEIGHT, IMAGE_WIDTH = 128, 128
DATASET_DIR = "C:/Users/USR-TSD-PC 05/Skripsi/Aktivitas/"
CLASSES_LIST = ["Berdiri", "Berjalan", "Duduk", "Jatuh"]


✅ GPU terdeteksi: /physical_device:GPU:0


In [4]:
# ===== Cell 3: Mediapipe & Landmark Grouping =====
mp_pose = mp.solutions.pose

# Landmark grouping definitions
HEAD_LANDMARKS       = list(range(0, 11))
BODY_LANDMARKS       = [11, 12]
LEFT_HAND_LANDMARKS  = [13, 15, 17, 19, 21]
RIGHT_HAND_LANDMARKS = [14, 16, 18, 20, 22]
LEFT_LEG_LANDMARKS   = [23, 25, 27, 29, 31]
RIGHT_LEG_LANDMARKS  = [24, 26, 28, 30, 32]

BLOCKED_CONNECTION_GROUPS = []

GROUP_MARKER = {
    'head': cv2.MARKER_TRIANGLE_UP, 'body': cv2.MARKER_DIAMOND,
    'left_hand': cv2.MARKER_TRIANGLE_DOWN, 'right_hand': cv2.MARKER_STAR,
    'left_leg': cv2.MARKER_SQUARE, 'right_leg': cv2.MARKER_TILTED_CROSS,
}
GROUP_COLOR = {
    'head': (0,255,255), 'body': (255,0,0),
    'left_hand': (0,255,0), 'right_hand': (0,200,0),
    'left_leg': (0,0,255), 'right_leg': (255,0,255),
}

def get_landmark_group(idx):
    if idx in HEAD_LANDMARKS:       return 'head'
    if idx in BODY_LANDMARKS:       return 'body'
    if idx in LEFT_HAND_LANDMARKS:  return 'left_hand'
    if idx in RIGHT_HAND_LANDMARKS: return 'right_hand'
    if idx in LEFT_LEG_LANDMARKS:   return 'left_leg'
    if idx in RIGHT_LEG_LANDMARKS:  return 'right_leg'
    return 'unknown'


In [5]:
def compute_head_centroid(landmarks, w, h):
    xs = [landmarks[i].x * w for i in HEAD_LANDMARKS if i < len(landmarks)]
    ys = [landmarks[i].y * h for i in HEAD_LANDMARKS if i < len(landmarks)]
    if not xs or not ys:
        return None
    return (int(sum(xs) / len(xs)), int(sum(ys) / len(ys)))


def draw_vertical_zigzag(canvas, amplitude=20, period=40, thickness=2, x_pos='center'):
    h, w = canvas.shape[:2]
    
    if isinstance(x_pos, str):
        if x_pos == 'left':
            cx = int(w * 0.1)
        elif x_pos == 'right':
            cx = int(w * 0.9)
        else:
            cx = w // 2
    else:
        cx = int(x_pos)  # langsung pakai nilai numerik

    pts, y, direction = [], 0, 1
    while y <= h:
        pts.append((cx + direction * amplitude, y))
        y += period
        direction *= -1
    for i in range(len(pts) - 1):
        cv2.line(canvas, pts[i], pts[i + 1], (0, 255, 255), thickness)


def draw_landmarks(canvas, landmarks, w, h):
    for i in HEAD_LANDMARKS:
        if i < len(landmarks):
            lm = landmarks[i]
            x, y = int(lm.x * w), int(lm.y * h)
            cv2.drawMarker(canvas, (x, y), GROUP_COLOR['head'],
                           markerType=GROUP_MARKER['head'], markerSize=7, thickness=2)
    for i, lm in enumerate(landmarks):
        if i in HEAD_LANDMARKS:
            continue
        x, y = int(lm.x * w), int(lm.y * h)
        g = get_landmark_group(i)
        cv2.drawMarker(canvas, (x, y), GROUP_COLOR.get(g, (128, 128, 128)),
                       markerType=GROUP_MARKER.get(g, cv2.MARKER_CROSS), markerSize=15, thickness=2)


def draw_connections(canvas, landmarks, w, h):
    def interp(p1, p2, steps, marker, color):
        x1, y1 = int(p1.x * w), int(p1.y * h)
        x2, y2 = int(p2.x * w), int(p2.y * h)
        for t in range(1, steps):
            alpha = t / steps
            xi = int(x1 * (1 - alpha) + x2 * alpha)
            yi = int(y1 * (1 - alpha) + y2 * alpha)
            cv2.drawMarker(canvas, (xi, yi), color,
                           markerType=marker, markerSize=10, thickness=2)
    for a, b in mp_pose.POSE_CONNECTIONS:
        if a >= len(landmarks) or b >= len(landmarks):
            continue
        g1, g2 = get_landmark_group(a), get_landmark_group(b)
        if g1 == 'head' and g2 == 'head':
            continue
        if any(a in grp and b in grp for grp in BLOCKED_CONNECTION_GROUPS):
            continue
        p1, p2 = landmarks[a], landmarks[b]
        if g1 == g2 != 'unknown':
            interp(p1, p2, 5, GROUP_MARKER[g1], GROUP_COLOR[g1])
        elif (a, b) in [(11, 23), (12, 24)]:
            m = GROUP_MARKER.get(g1, cv2.MARKER_CROSS)
            c = GROUP_COLOR.get(g1, (128, 128, 128))
            interp(p1, p2, 9, m, c)


def crop_and_norm(frame, landmarks, w, h):
    xs = [lm.x for lm in landmarks]
    ys = [lm.y for lm in landmarks]
    if not xs or not ys:
        return None
    x1, y1 = int(min(xs) * w) - 20, int(min(ys) * h) - 20
    x2, y2 = int(max(xs) * w) + 20, int(max(ys) * h) + 20
    x1, y1 = max(x1, 0), max(y1, 0)
    x2, y2 = min(x2, w), min(y2, h)
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    return cv2.resize(crop, (IMAGE_HEIGHT, IMAGE_WIDTH)).astype(np.float32)/255.0


def augment_sequence(sequence):
    # sequence adalah list frames (numpy array)
    seq_orig = sequence
    seq_flip_v = [cv2.flip(f, 0) for f in sequence]  # flip vertical
    seq_flip_h = [cv2.flip(f, 1) for f in sequence]  # flip horizontal
    seq_flip_vh = [cv2.flip(f, -1) for f in sequence] # flip vertical+horizontal
    
    return [seq_orig, seq_flip_v, seq_flip_h, seq_flip_vh]


def add_zigzag_to_sequence(sequence, amplitude, period, thickness=2, x_pos='center'):
    result = []
    for frame in sequence:
        frame_copy = frame.copy()
        draw_vertical_zigzag(frame_copy, amplitude=amplitude, period=period, thickness=thickness, x_pos=x_pos)
        result.append(frame_copy)
    return result


def frames_extraction_with_zigzag(video_path, visualize=False, max_samples=0, sequence_length=6, return_raw=False):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return [] if not return_raw else ([], [])

    raw_records = []  # list of (canvas, landmarks, w, h)
    raw_centroids = []
    raw_imgs = []  # SIMPAN frame asli di sini

    with mp_pose.Pose(static_image_mode=False, model_complexity=1,
                      enable_segmentation=False, min_detection_confidence=0.5) as pose:
        while True:
            ret, img = cap.read()
            if not ret:
                break
            h, w = img.shape[:2]
            res = pose.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if not res.pose_landmarks:
                continue
            lm = res.pose_landmarks.landmark
            canvas = np.zeros_like(img)
            draw_landmarks(canvas, lm, w, h)
            draw_connections(canvas, lm, w, h)
            cent = compute_head_centroid(lm, w, h)
            raw_centroids.append(cent)
            raw_records.append((canvas, lm, w, h))
            raw_imgs.append(img.copy())

    cap.release()

    if len(raw_records) < sequence_length:
        return [] if not return_raw else ([], [])

    idxs = np.linspace(0, len(raw_records) - 1, sequence_length, dtype=int)
    seq = [raw_records[i] for i in idxs]
    cent_seq = [raw_centroids[i] for i in idxs]
    raw_seq_imgs = [raw_imgs[i] for i in idxs]

    c1, c2 = cent_seq[0], cent_seq[sequence_length // 2]
    dx = abs(c2[0] - c1[0]) if c1 and c2 else 0
    dy = abs(c2[1] - c1[1]) if c1 and c2 else 0
    use_zigzag = dx >= 170 or dy >= 160

    total_w = IMAGE_WIDTH
    gap_w = int(total_w * 0.05)
    pose_w = int(total_w * 0.85)
    zigzag_w = total_w - pose_w - gap_w  # sisanya buat area kosong zigzag

    # Buat dua versi sequence: crop pose di kiri & crop pose di kanan
    seq_frames_left = []   # untuk original & vertical flip
    seq_frames_right = []  # untuk horizontal & horizontal+vertical flip

    for canvas, lm, rw, rh in seq:
        pose_crop = crop_and_norm(canvas, lm, rw, rh)
        if pose_crop is None:
            continue
        pose_crop_uint8 = (pose_crop * 255).astype(np.uint8)

        # --- versi kiri (pose di kiri, kosong di kanan) ---
        pose_img_left = cv2.resize(pose_crop_uint8, (pose_w, IMAGE_HEIGHT))
        gap_img = np.zeros((IMAGE_HEIGHT, gap_w, 3), dtype=np.uint8)
        zigzag_area = np.zeros((IMAGE_HEIGHT, zigzag_w, 3), dtype=np.uint8)
        combined_left = np.concatenate([pose_img_left, gap_img, zigzag_area], axis=1)
        seq_frames_left.append(combined_left)

        # --- versi kanan (kosong di kiri, pose di kanan) ---
        zigzag_area2 = np.zeros((IMAGE_HEIGHT, zigzag_w, 3), dtype=np.uint8)
        gap_img2 = np.zeros((IMAGE_HEIGHT, gap_w, 3), dtype=np.uint8)
        pose_img_right = cv2.resize(pose_crop_uint8, (pose_w, IMAGE_HEIGHT))
        combined_right = np.concatenate([zigzag_area2, gap_img2, pose_img_right], axis=1)
        seq_frames_right.append(combined_right)

    # Lakukan augmentasi sequence sekaligus
    augmented_sequences = []
    seq_aug_left = augment_sequence(seq_frames_left)
    seq_aug_right = augment_sequence(seq_frames_right)

    # original -> versi kiri
    augmented_sequences.append(seq_aug_left[0])
    # vertical flip -> versi kiri
    augmented_sequences.append(seq_aug_left[1])
    # horizontal flip -> versi kanan
    augmented_sequences.append(seq_aug_right[2])
    # horizontal + vertical flip -> versi kanan
    augmented_sequences.append(seq_aug_right[3])

    # Jika zigzag digunakan, tambahkan zigzag ke seluruh frame di tiap sequence augmentasi
    if use_zigzag:
        amp = max(1, zigzag_w // 2)
        per = max(1, IMAGE_HEIGHT // 6)

        final_augmented_sequences = []
        for i, seq_aug in enumerate(augmented_sequences):
            if i in [0, 1]:  # original & vertical flip -> zigzag kanan
                pos = 'right'
            else:            # horizontal & horizontal+vertical flip -> zigzag kiri
                pos = 'right'

            seq_with_zigzag = add_zigzag_to_sequence(seq_aug, amp, per, thickness=2, x_pos=pos)
            final_augmented_sequences.append(seq_with_zigzag)

        augmented_sequences = final_augmented_sequences

    # Normalisasi kembali (0..1)
    augmented_sequences = [[frame.astype(np.float32)/255.0 for frame in seq_aug] for seq_aug in augmented_sequences]

    # Gabungkan semua frame dari semua augmentasi menjadi satu list
    all_frames = []
    for seq_aug in augmented_sequences:
        all_frames.extend(seq_aug)

    return (all_frames, raw_seq_imgs) if return_raw else all_frames


In [6]:
import os

def create_dataset_paths(dataset_dir, classes_list):
    """
    Kembalikan list tuple (video_path, class_idx), tanpa augmentasi atau ekstraksi.
    """
    data_paths = []
    for class_idx, class_name in enumerate(classes_list):
        class_dir = os.path.join(dataset_dir, class_name)
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.mp4', '.avi', '.mov')):
                video_path = os.path.join(class_dir, fname)
                data_paths.append((video_path, class_idx))
    return data_paths

# Contoh pakai:
video_paths = create_dataset_paths(DATASET_DIR, CLASSES_LIST)
print(f"Total videos found: {len(video_paths)}")

from sklearn.model_selection import train_test_split

# Split 80% train, 20% test (misal)
train_paths, test_paths = train_test_split(
    video_paths,
    test_size=0.2,
    stratify=[label for _, label in video_paths],
    random_state=50
)

# Dari train, split lagi 20% jadi val
train_paths, val_paths = train_test_split(
    train_paths,
    test_size=0.2,
    stratify=[label for _, label in train_paths],
    random_state=50
)

print(f"Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")


def process_dataset(paths, sequence_length, return_raw=False):
    """
    Proses video paths yang sudah di-split, lakukan ekstraksi frame dan augmentasi.
    """
    X, y = [], []
    raw_data = [] if return_raw else None

    for video_path, class_idx in paths:
        # Ekstrak frame dan augmentasi di sini
        if return_raw:
            frames, raw_frames = frames_extraction_with_zigzag(
                video_path,
                visualize=False,
                max_samples=0,
                sequence_length=sequence_length,
                return_raw=True
            )
        else:
            frames = frames_extraction_with_zigzag(
                video_path,
                visualize=False,
                max_samples=0,
                sequence_length=sequence_length
            )

        if len(frames) < sequence_length:
            continue

        num_sequences = len(frames) // sequence_length
        for i in range(num_sequences):
            start = i * sequence_length
            end = start + sequence_length
            sequence = frames[start:end]

            if len(sequence) == sequence_length:
                X.append(sequence)
                y.append(class_idx)
                if return_raw:
                    raw_data.append(raw_frames)

    if return_raw:
        return np.array(X), np.array(y), raw_data
    return np.array(X), np.array(y)


# Contoh proses data:
X_train, y_train = process_dataset(train_paths, sequence_length=6)
X_val, y_val = process_dataset(val_paths, sequence_length=6)
X_test, y_test = process_dataset(test_paths, sequence_length=6)

print(f"Train samples: {len(X_train)}")
print(f"Val samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")


Total videos found: 100
Train: 64, Val: 16, Test: 20
Train samples: 256
Val samples: 64
Test samples: 80


In [7]:
# ===== Cell 6: Model Definition =====
def build_model(sequence_length):
    inp_shape = (sequence_length, IMAGE_HEIGHT, IMAGE_WIDTH, 3)
    m = Sequential([
        TimeDistributed(Conv2D(16,(5,5),padding='same',activation='relu'), input_shape=inp_shape),
        TimeDistributed(BatchNormalization()),
        TimeDistributed(MaxPooling2D((4,4))),
        TimeDistributed(Dropout(0.25)),

        TimeDistributed(Conv2D(16,(5,5),padding='same',activation='relu')),
        TimeDistributed(BatchNormalization()),
        TimeDistributed(MaxPooling2D((4,4))),
        TimeDistributed(Dropout(0.25)),

        TimeDistributed(Conv2D(64,(5,5),padding='same',activation='relu')),
        TimeDistributed(BatchNormalization()),
        TimeDistributed(MaxPooling2D((4,4))),
        TimeDistributed(Dropout(0.25)),

        TimeDistributed(Conv2D(64,(5,5),padding='same',activation='relu')),
        TimeDistributed(MaxPooling2D((2,2))),
        TimeDistributed(Flatten()),

        LSTM(64),
        Dense(32, activation='relu'),
        Dense(len(CLASSES_LIST), activation='softmax')
    ])
    m.compile(loss='categorical_crossentropy',
              optimizer=Adam(1e-4),
              metrics=['accuracy'])
    return m


In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import cv2

# ===== Cell 7: Training Loop & Experiments =====
# ===== Cell Training Loop & Experiments =====
sequence_lengths = [6]
results = {}

for seq_len in sequence_lengths:
    print(f"\n>> EXPERIMENT: sequence_length = {seq_len}")
    
    # Load & split (include raw untuk test)
    X_train, y_train = process_dataset(train_paths, seq_len)
    X_val, y_val = process_dataset(val_paths, seq_len)
    X_test, y_test, raw_test = process_dataset(test_paths, seq_len, return_raw=True)

    # One-hot encoding
    y_train = to_categorical(y_train, num_classes=len(CLASSES_LIST))
    y_val = to_categorical(y_val, num_classes=len(CLASSES_LIST))
    y_test = to_categorical(y_test, num_classes=len(CLASSES_LIST))

    # Visualisasi sampel
    #show_sample_sequence(X_train, y_train, CLASSES_LIST, f"Train Seq={seq_len}")
    #show_sample_sequence(X_test,  y_test,  CLASSES_LIST, f"Test Seq={seq_len}")

    # Build model & callbacks
    model = build_model(seq_len)
    print(model.summary())

    # ======= Train with loop until accuracy >= TARGET_ACC =======
    MAX_TRIALS = 10
    TARGET_ACC = 1

    best_model = None
    best_acc = 0.0
    best_hist = None
    best_y_pred = None
    best_y_true = None

    # (Optional) Custom learning rate scheduler callback jika ingin dipakai
    class CustomLRScheduler(Callback):
        def __init__(self, decay_factor=0.9):
            super().__init__()
            self.decay_factor = decay_factor

        def on_epoch_end(self, epoch, logs=None):
            old_lr = float(tf.keras.backend.get_value(self.model.optimizer.lr))
            new_lr = old_lr * self.decay_factor
            tf.keras.backend.set_value(self.model.optimizer.lr, new_lr)
            print(f"\nEpoch {epoch+1}: Learning rate updated from {old_lr:.6f} to {new_lr:.6f}")

    for trial in range(1, MAX_TRIALS + 1):
        print(f"\n🔁 Training attempt #{trial}")

        model = build_model(seq_len)

        ckpt = ModelCheckpoint(f"temp_best_seq{seq_len}.h5", save_best_only=True, monitor='val_accuracy')
        early_stop = EarlyStopping(monitor='val_loss', patience=65, restore_best_weights=True, verbose=1)
        rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.875, patience=15, verbose=1)
        tb = TensorBoard(log_dir=f"logs/seq{seq_len}_trial{trial}")

        hist = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=99999,  # early stopping akan mengatur berhentinya
            batch_size=6,
            callbacks=[early_stop, rlr, tb],  # bisa tambahkan CustomLRScheduler() jika mau
            verbose=1
        )

        y_pred = np.argmax(model.predict(X_test), axis=1)
        y_true = np.argmax(y_test, axis=1)
        acc = accuracy_score(y_true, y_pred)

        print(f"🎯 Accuracy on test: {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            best_model = model
            best_hist = hist
            best_y_pred = y_pred
            best_y_true = y_true

        if acc >= TARGET_ACC:
            print("✅ Target accuracy achieved.")
            break
        else:
            print("❌ Target not achieved, retrying...")

    # Set hasil terbaik untuk evaluasi
    model = best_model
    hist = best_hist
    y_pred = best_y_pred
    y_true = best_y_true

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Per-class metrics
    class_metrics = []
    for i, class_name in enumerate(CLASSES_LIST):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

        class_metrics.append([class_name, precision, recall, specificity])

    class_metrics = np.array(class_metrics, dtype=object)

    # Hitung rata-rata makro
    acc = accuracy_score(y_true, y_pred)
    prec_macro = np.mean(class_metrics[:,1].astype(float))
    rec_macro = np.mean(class_metrics[:,2].astype(float))
    spec_macro = np.mean(class_metrics[:,3].astype(float))

    results[seq_len] = {
        'accuracy': acc,
        'precision': prec_macro,
        'recall': rec_macro,
        'specificity': spec_macro
    }

    print(f"-> Accuracy    : {acc:.4f}")
    print(f"-> Precision   : {prec_macro:.4f}")
    print(f"-> Sensitivity : {rec_macro:.4f}")
    print(f"-> Specificity : {spec_macro:.4f}")

    # ===== TABEL METRIK PER KELAS =====
    fig, ax = plt.subplots(figsize=(8, len(CLASSES_LIST)*0.5 + 2))
    table_data = [["Class", "Precision", "Recall (Sensitivity)", "Specificity"]] + class_metrics.tolist()
    table = ax.table(cellText=table_data, loc='center', cellLoc='center')
    table.scale(1, 2)
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    ax.axis('off')
    plt.title("Per-Class Evaluation Metrics", fontsize=14)
    plt.show()

    # Simpan model dengan nama konsisten
    model_filename = f"augmentasi_zigzag_seq{seq_len}_acc_100%.h5"
    model.save(model_filename)
    print(f"💾 Model disimpan di {model_filename}")

    # Plot training history
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(hist.history['accuracy'], label='accuracy')
    plt.plot(hist.history['val_accuracy'], label='val_accuracy')
    plt.legend()
    plt.title("Accuracy")
    plt.xlabel("Epoch")

    plt.subplot(1, 2, 2)
    plt.plot(hist.history['loss'], label='loss')
    plt.plot(hist.history['val_loss'], label='val_loss')
    plt.legend()
    plt.title("Loss")
    plt.xlabel("Epoch")

    plt.tight_layout()
    plt.show()

    # Confusion matrix plot
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES_LIST, yticklabels=CLASSES_LIST, cmap='Blues')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.show()

    # Simpan raw test data
    raw_data_filename = f"raw_test_seq{seq_len}.pkl"
    with open(raw_data_filename, "wb") as f:
        pickle.dump({
            "raw_test": raw_test,
            "y_true": y_true,
            "y_pred": y_pred
        }, f)
    print(f"📦 Raw test data disimpan di {raw_data_filename}")

# === Visualisasi SEMUA data test ===
print("\n📊 Menampilkan seluruh prediksi di test set:")

for i in range(len(y_true)):
    label_true = CLASSES_LIST[y_true[i]]
    label_pred = CLASSES_LIST[y_pred[i]]
    color = "green" if y_true[i] == y_pred[i] else "red"
    n_seq = len(raw_test[i])

    plt.figure(figsize=(15, 4), dpi=150)
    
    # Raw frames
    for j, frame in enumerate(raw_test[i]):
        plt.subplot(2, n_seq, j + 1)
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        if j == 0:
            plt.ylabel("Raw", fontsize=10)

    # Preprocessed frames
    for j in range(n_seq):
        plt.subplot(2, n_seq, n_seq + j + 1)
        plt.imshow(X_test[i][j])
        plt.axis('off')
        if j == 0:
            plt.ylabel("Preprocessed", fontsize=10)

    plt.suptitle(
        f"[{'BENAR' if y_true[i]==y_pred[i] else 'SALAH'}] Index: {i} | True = {label_true} | Pred = {label_pred}",
        color=color, fontsize=14
    )
    plt.tight_layout()
    plt.show()


# Summary hasil akhir
print("\n=== Final Results ===")
for sl, metrics in results.items():
    print(f"\nSequence {sl}:")
    print(f"  Accuracy    = {metrics['accuracy']:.4f}")
    print(f"  Precision   = {metrics['precision']:.4f}")
    print(f"  Sensitivity = {metrics['recall']:.4f}")
    print(f"  Specificity = {metrics['specificity']:.4f}")

In [ ]:
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import pickle
import numpy as np
import cv2

# ===== Cell 7: Training Loop & Experiments =====
# ===== Cell Training Loop & Experiments =====
sequence_lengths = [6]
results = {}

for seq_len in sequence_lengths:
    print(f"\n>> EXPERIMENT: sequence_length = {seq_len}")
    
    # Load & split (include raw untuk test)
    X_train, y_train = process_dataset(train_paths, seq_len)
    X_val, y_val = process_dataset(val_paths, seq_len)
    X_test, y_test, raw_test = process_dataset(test_paths, seq_len, return_raw=True)

    # One-hot encoding
    y_test_cat = to_categorical(y_test, len(CLASSES_LIST))

    # Load modelC:\Users\USR-TSD-PC 05\Skripsi\augmentasi_zigzag_seq6_acc_98.75%.h5
    model_path = "C:/Users/USR-TSD-PC 05/Skripsi/augmentasi_zigzag_seq6_acc_100%.h5"
    model = load_model(model_path)
    print(f"📥 Model loaded from {model_path}")
    print(model.summary())

    # Predict
    y_probs = model.predict(X_test)
    y_pred = np.argmax(y_probs, axis=1)
    y_true = y_test
    confidence_scores = np.max(y_probs, axis=1)

    # Loss per sample
    loss_fn = tf.keras.losses.CategoricalCrossentropy(reduction=tf.keras.losses.Reduction.NONE)
    losses = loss_fn(y_test_cat, y_probs).numpy()

    # Evaluate
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    # Metrics per class
    class_metrics = []
    for i, class_name in enumerate(CLASSES_LIST):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

        class_metrics.append([class_name, precision, recall, specificity])

    class_metrics = np.array(class_metrics, dtype=object)
    prec_macro = np.mean(class_metrics[:, 1].astype(float))
    rec_macro = np.mean(class_metrics[:, 2].astype(float))
    spec_macro = np.mean(class_metrics[:, 3].astype(float))

    results[seq_len] = {
        'accuracy': acc,
        'precision': prec_macro,
        'recall': rec_macro,
        'specificity': spec_macro
    }

    print(f"-> Accuracy    : {acc:.4f}")
    print(f"-> Precision   : {prec_macro:.4f}")
    print(f"-> Sensitivity : {rec_macro:.4f}")
    print(f"-> Specificity : {spec_macro:.4f}")

    # Confusion Matrix
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES_LIST, yticklabels=CLASSES_LIST, cmap='Blues')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.show()

    # === 🔄 Visualisasi gabungan per sampel ===
    for i in range(len(y_true)):
        label_true = CLASSES_LIST[y_true[i]]
        label_pred = CLASSES_LIST[y_pred[i]]
        confidence = confidence_scores[i]
        loss = losses[i]
        probas = y_probs[i]
        n_seq = len(raw_test[i])

        fig = plt.figure(figsize=(20, 8), dpi=150)
        grid = plt.GridSpec(3, n_seq, hspace=0.6)

        # === ROW 1: RAW FRAMES
        for j, frame in enumerate(raw_test[i]):
            ax_raw = fig.add_subplot(grid[0, j])
            ax_raw.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax_raw.axis('off')
            if j == 0:
                ax_raw.set_ylabel("Raw", fontsize=12)

        # === ROW 2: PREPROCESSED FRAMES
        for j in range(n_seq):
            ax_proc = fig.add_subplot(grid[1, j])
            ax_proc.imshow(X_test[i][j])
            ax_proc.axis('off')
            if j == 0:
                ax_proc.set_ylabel("Preprocessed", fontsize=12)

        # === ROW 3: HORIZONTAL BAR CHART (Confidence)
        ax_bar = fig.add_subplot(grid[2, :])
        bars = ax_bar.barh(range(len(CLASSES_LIST)), probas, color='skyblue')
        bars[y_pred[i]].set_color('orange')  # prediksi
        bars[y_true[i]].set_color('green')   # label asli
        ax_bar.set_yticks(range(len(CLASSES_LIST)))
        ax_bar.set_yticklabels(CLASSES_LIST)
        ax_bar.set_xlim(0, 1)
        ax_bar.set_xlabel("Confidence")
        ax_bar.set_title(
            f"Sampel #{i+1} | True: {label_true} | Pred: {label_pred} | "
            f"Confidence: {confidence:.2f} | Loss: {loss:.4f}",
            fontsize=12
        )
        ax_bar.grid(True, axis='x', linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.show()


    # Save evaluation results
    with open(f"pose_block_results_with_confidence_seq{seq_len}.pkl", "wb") as f:
        pickle.dump({
            "raw_test": raw_test,
            "y_true": y_true,
            "y_pred": y_pred,
            "confidence": confidence_scores,
            "losses": losses
        }, f)
    print(f"💾 Data hasil disimpan di pose_block_results_with_confidence_seq{seq_len}.pkl")

# Final summary
print("\n=== Final Results ===")
for sl, metrics in results.items():
    print(f"\nSequence {sl}:")
    print(f"  Accuracy    = {metrics['accuracy']:.4f}")
    print(f"  Precision   = {metrics['precision']:.4f}")
    print(f"  Sensitivity = {metrics['recall']:.4f}")
    print(f"  Specificity = {metrics['specificity']:.4f}")